In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import logging

# --- 1. SILENCE WARNINGS ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok
import numpy as np
import joblib
import pywt
import shap
import pandas as pd
import ast
from scipy.spatial.distance import euclidean
from tensorflow.keras.models import load_model
from collections import Counter
from fastapi.middleware.cors import CORSMiddleware

# --- Initialize Server ---
nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- 2. CONFIGURATION & MODEL LOADING ---
WAVELET = "db4"
LEVEL = 7
SFREQ = 200
SCALER_PATH = "/content/drive/My Drive/scaler_wavelet_1.pkl"
MODEL_PATH = "/content/drive/My Drive/best_strict_model.keras"
LIB_PATH = '/content/drive/My Drive/EEG_Prototype_Library/signature_index.csv'

print("Initializing WAVESAGE Backend...")
scaler = joblib.load(SCALER_PATH)
model = load_model(MODEL_PATH)
lib_df = pd.read_csv(LIB_PATH)

background = np.load("/content/drive/My Drive/background_data_2.npy")
explainer = shap.DeepExplainer(model, background)

top_percent_by_coeff = {"Coeff 5": 0.25, "Coeff 4": 0.25, "Coeff 3": 0.25, "Coeff 2": 0.25, "Coeff 1": 0.25}
thresholds_by_coeff = {"Coeff 5": 1e-10, "Coeff 4": 3e-9, "Coeff 3": 5e-9, "Coeff 2": 5e-8, "Coeff 1": 5e-8}

# --- 3. CORE LOGIC (Simplified for readability) ---

def build_coeff_slices(coeffs):
    slices, start = {}, 0
    for i, c in enumerate(coeffs):
        end = start + len(c)
        slices[f"Coeff {i}"] = (start, end)
        start = end
    return slices

def merge_segments(segments, gap=0.05):
    if not segments: return []
    merged = [segments[0]]
    for curr in segments[1:]:
        if curr[0] - merged[-1][1] <= gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], curr[1]))
        else: merged.append(curr)
    return merged

def majority_vote_segments(segments_dict, k=3):
    all_points = []
    for segs in segments_dict.values():
        for s, e in segs:
            all_points.extend([(s, 'start'), (e, 'end')])
    all_points.sort()
    active, merged, seg_start = 0, [], None
    for t, typ in all_points:
        if typ == 'start':
            active += 1
            if active == k: seg_start = t
        else:
            if active == k and seg_start is not None: merged.append((seg_start, t))
            active -= 1
    return merge_segments(merged, gap=0.1)

def extract_signatures(segment, sfreq):
    coeffs = pywt.wavedec(segment, 'db4', level=7)
    d3_to_d7 = coeffs[1:6]
    energies = [np.sum(np.square(c)) for c in d3_to_d7]
    rel_energy = np.array(energies) / np.sum(energies) if np.sum(energies) > 0 else np.zeros(5)
    return {"rel_energy_d3_d7": rel_energy.tolist(), "amplitude": float(np.ptp(segment)), "duration": float(len(segment)/sfreq)}

def identify_wave_type(query_segment, sfreq, library_df, k=5):
    query_sig = extract_signatures(query_segment, sfreq)
    query_vec = np.array(query_sig['rel_energy_d3_d7'])
    all_neighbors = []
    for _, row in library_df.iterrows():
        lib_vec = np.array(ast.literal_eval(row['rel_energy_d3_d7']))
        dist = euclidean(query_vec, lib_vec)
        all_neighbors.append((row['wave_label'], dist + (abs(query_sig['duration'] - row['duration']) * 0.5)))
    all_neighbors.sort(key=lambda x: x[1])
    winner_label = Counter([n[0] for n in all_neighbors[:k]]).most_common(1)[0][0]
    avg_dist = np.mean([n[1] for n in all_neighbors[:k] if n[0] == winner_label])
    return winner_label, avg_dist

def run_wavesage_logic(window_raw):
    full_coeffs = pywt.wavedec(window_raw, WAVELET, level=LEVEL)
    coeffs_used = full_coeffs[:6]
    features = np.concatenate(coeffs_used)
    coeff_slices = build_coeff_slices(coeffs_used)
    features_scaled = scaler.transform([features]).reshape(1, -1, 1)
    shap_vals = explainer.shap_values(features_scaled)
    shap_pos = np.maximum(shap_vals[0].reshape(-1), 0)
    important_mask = np.zeros_like(shap_pos, dtype=bool)
    for name, (start, end) in coeff_slices.items():
        pct = top_percent_by_coeff.get(name, 0.20)
        coeff_shap = shap_pos[start:end]
        k = max(1, int(len(coeff_shap) * pct))
        important_mask[np.argsort(coeff_shap)[-k:] + start] = True
    time_axis = np.linspace(0, 2, len(window_raw))
    segments_by_coeff = {}
    for c_id in [5, 4, 3, 2, 1]:
        name = f"Coeff {c_id}"
        c_mask = important_mask[coeff_slices[name][0]:coeff_slices[name][1]]
        recon_coeffs = [np.zeros_like(c) for c in full_coeffs]
        recon_coeffs[c_id][c_mask] = full_coeffs[c_id][c_mask]
        reconstructed = pywt.waverec(recon_coeffs, WAVELET)[:len(time_axis)]
        nz_mask = np.abs(reconstructed) > thresholds_by_coeff.get(name, 1e-9)
        segs, in_seg, start_i = [], False, 0
        for i, val in enumerate(nz_mask):
            if val and not in_seg: start_i, in_seg = i, True
            elif not val and in_seg: segs.append((time_axis[start_i], time_axis[i])); in_seg = False
        segments_by_coeff[c_id] = merge_segments(segs, gap=0.1)
    return majority_vote_segments(segments_by_coeff, k=3)

# --- 4. API ENDPOINT WITH REAL-TIME PRINTING ---

@app.post("/analyze_eeg")
async def analyze_eeg(info: Request):
    data = await info.json()
    all_signals = np.array(data.get('signals', []))
    channel_names = data.get('channel_names', [])
    win_size = int(2 * SFREQ)
    all_results = []

    print(f"\n--- Processing Data Chunk ({len(all_signals)} channels) ---")

    for ch_idx, signal in enumerate(all_signals):
        ch_name = channel_names[ch_idx] if ch_idx < len(channel_names) else f"CH_{ch_idx}"

        for i in range(0, len(signal) - win_size, win_size):
            window = signal[i : i + win_size]
            offset = i / SFREQ
            feat = np.concatenate(pywt.wavedec(window, WAVELET, level=LEVEL)[:6])
            feat_scaled = scaler.transform([feat]).reshape(1, -1, 1)

            # 1. Get Prediction Value
            pred_prob = float(model.predict(feat_scaled, verbose=0)[0][0])

            # 2. Print Real-time Prediction
            status = "✅ DETECTION" if pred_prob > 0.999 else "⚪ NO EVENT"
            print(f"[{ch_name}] Time: {offset:5.2f}s | Pred: {pred_prob:7.4f} | {status}")

            if pred_prob > 0.999:
                loc_segs = run_wavesage_logic(window)
                for s, e in loc_segs:
                    crop = window[int(s*SFREQ):int(e*SFREQ)]
                    if len(crop) < 5: continue
                    label, dist = identify_wave_type(crop, SFREQ, lib_df)
                    all_results.append({
                        "start_time": offset + s, "end_time": offset + e,
                        "label": label.upper(), "prediction_score": pred_prob,
                        "channel_name": ch_name
                    })

    return {"status": "success", "events": all_results}

# --- 5. START SERVER ---
auth_token = "2kNBEOUr2bfA2rKeh1t0cLHFKa0_6BtaUX3ZU33GLvHzzmMZv"
ngrok.set_auth_token(auth_token)
public_url = ngrok.connect(8000)
print(f"--- BACKEND LIVE AT: {public_url} ---")

if __name__ == "__main__":
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
    server = uvicorn.Server(config)
    await server.serve()

Initializing WAVESAGE Backend...
--- BACKEND LIVE AT: NgrokTunnel: "https://88aa-34-91-155-21.ngrok-free.app" -> "http://localhost:8000" ---

--- Processing Data Chunk (6 channels) ---
[FP1] Time:  0.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time:  2.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time:  4.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time:  6.00s | Pred:  1.0000 | ✅ DETECTION
[FP1] Time:  8.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time: 10.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time: 12.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time: 14.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP1] Time: 16.00s | Pred:  1.0000 | ✅ DETECTION
[FP2] Time:  0.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time:  2.00s | Pred:  1.0000 | ✅ DETECTION
[FP2] Time:  4.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time:  6.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time:  8.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time: 10.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time: 12.00s | Pred:  0.0000 | ⚪ NO EVENT
[FP2] Time: 14.00s | Pred:  1.0000 | ✅ DETEC